In [2]:
import torch
import torch.nn as nn

class LSTMSequenceClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super(LSTMSequenceClassifier, self).__init__()
        
        # 1. Embedding Layer: Converts integer word tokens into dense continuous vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # 2. LSTM Engine
        # batch_first=True configures the layer input shape to: (Batch, Sequence Length, Features)
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim, num_layers=1, batch_first=True)
        
        # 3. Output classification layer
        self.fc = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x):
        # Input shape x: (Batch, Seq_Len)
        embedded = self.embedding(x) # Shape transitions to: (Batch, Seq_Len, Embedding_Dim)
        
        # Pass through the LSTM engine
        # out: contains hidden states for EVERY time step in the sequence
        # (hn, cn): contain the final, absolute terminal hidden state and cell state tensors
        out, (hn, cn) = self.lstm(embedded)
        
        # For sequence classification, we extract the very last hidden state output 
        # representing the cumulative memory accumulated over the entire timeline.
        # hn shape: (num_layers, batch, hidden_dim) -> squeeze layer dim out
        final_hidden_state = hn[-1] 
        
        # Map structural memory directly to target class logits
        logits = self.fc(final_hidden_state)
        return logits

# Instantiate the sequence model
# Imagine an NLP task with 5000 unique vocabulary words, mapping to a 3-class sentiment target
model = LSTMSequenceClassifier(vocab_size=5000, embedding_dim=128, hidden_dim=256, num_classes=3)
print(model)

# Verify tensor tracking arrays with a mock text batch (4 sentences, each containing exactly 15 tokens)
mock_text_batch = torch.randint(low=0, high=5000, size=(4, 15))
output_logits = model(mock_text_batch)

print(f"\nForward sequence tracking successful!")
print(f"Output Matrix Shape: {output_logits.shape} (Batch Size, Target Sentiment Classes)")

LSTMSequenceClassifier(
  (embedding): Embedding(5000, 128)
  (lstm): LSTM(128, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=3, bias=True)
)

Forward sequence tracking successful!
Output Matrix Shape: torch.Size([4, 3]) (Batch Size, Target Sentiment Classes)
